# Lab 17 — Robustness Mitigation Comparison

*AI Cybersecurity Masterclass · Lab 17 · AI60secLab*

**Companion video:** [▶ Making Security AI Harder to Fool](https://www.youtube.com/watch?v=ug9UqkDUgEo&list=PLG4l6Ssi9A00)

**Research question:** Which defensive treatment best preserves detection under controlled input stress without sacrificing too much clean performance?

We compare:
1. baseline;
2. validation-selected threshold;
3. bounded noisy training augmentation.

In [1]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

Project root: /home/claude/run/exec
Data folder : /home/claude/run/exec/data/raw


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.evaluation import binary_metrics
from src.thresholds import threshold_sweep, choose_threshold, apply_threshold
from src.adversarial import (
    available_numeric_attack_features,
    training_bounds,
    bounded_random_perturbation,
    augment_attacks_with_noise,
)

train, test = load_unsw(DATA_DIR)

X_full, y_full = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_full, X_test = align_columns(X_full, X_test)

X_full = drop_identifier_like_columns(X_full)
X_test = drop_identifier_like_columns(X_test)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    stratify=y_full,
    random_state=42,
)

In [3]:
baseline = random_forest_pipeline(X_train)
baseline.fit(X_train, y_train)

valid_scores = baseline.predict_proba(X_valid)[:, 1]
sweep = threshold_sweep(
    y_valid,
    valid_scores,
    beta=2.0,
)
selected_threshold, _ = choose_threshold(
    sweep,
    objective="f_beta",
)

print(
    "Validation-selected threshold:",
    round(selected_threshold, 3),
)

Validation-selected threshold: 0.21


In [4]:
perturbation_columns = available_numeric_attack_features(X_train)
bounds = training_bounds(X_train, perturbation_columns)

X_augmented, y_augmented = augment_attacks_with_noise(
    X_train,
    y_train,
    columns=perturbation_columns,
    epsilon=0.05,
    copies=1,
    bounds=bounds,
    random_state=42,
)

augmented_model = random_forest_pipeline(X_augmented)
augmented_model.fit(X_augmented, y_augmented)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](42,)","['dur','proto','service',...,'ct_src_ltm','ct_srv_dst','is_sm_ips_ports']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,42
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthro

In [5]:
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

X_stressed = bounded_random_perturbation(
    X_test,
    columns=perturbation_columns,
    epsilon=0.10,
    bounds=bounds,
    random_state=99,
)

rows = []

for condition, X_eval in [
    ("clean", X_test),
    ("stressed", X_stressed),
]:
    baseline_scores = baseline.predict_proba(X_eval)[:, 1]
    baseline_default = apply_threshold(
        baseline_scores, 0.50
    )
    baseline_tuned = apply_threshold(
        baseline_scores,
        selected_threshold,
    )

    augmented_scores = augmented_model.predict_proba(X_eval)[:, 1]
    augmented_pred = apply_threshold(
        augmented_scores, 0.50
    )

    for strategy, pred, score in [
        ("Baseline", baseline_default, baseline_scores),
        ("Threshold tuned", baseline_tuned, baseline_scores),
        ("Noise augmented", augmented_pred, augmented_scores),
    ]:
        metrics = binary_metrics(
            y_test,
            pred,
            score,
        )
        rows.append({
            "condition": condition,
            "strategy": strategy,
            "precision": metrics.precision,
            "recall": metrics.recall,
            "f1": metrics.f1,
            "false_negative_rate": metrics.false_negative_rate,
        })

comparison = pd.DataFrame(rows)
display(comparison.round(4))

,condition,strategy,precision,recall,f1,false_negative_rate
0,clean,Baseline,0.8149,0.9876,0.8930,0.0124
1,clean,Threshold tuned,0.7492,0.9992,0.8563,0.0008
2,clean,Noise augmented,0.8076,0.9901,0.8896,0.0099
3,stressed,Baseline,0.7977,0.9907,0.8838,0.0093
4,stressed,Threshold tuned,0.7443,0.9998,0.8533,0.0002
5,stressed,Noise augmented,0.7457,0.9998,0.8543,0.0002


In [6]:
pivot = comparison.pivot(
    index="strategy",
    columns="condition",
    values="recall",
)

pivot["recall_degradation"] = (
    pivot["clean"] - pivot["stressed"]
)

display(
    pivot.sort_values("recall_degradation").round(4)
)

condition,clean,stressed,recall_degradation
strategy,,,
Noise augmented,0.9901,0.9998,-0.0097
Baseline,0.9876,0.9907,-0.0031
Threshold tuned,0.9992,0.9998,-0.0006


## Interpretation

Which strategy had the strongest clean recall? Which had the smallest degradation? Did robustness improvement create a precision cost?

**Takeaway:** robustness mitigation is a tradeoff problem and must be evaluated under both clean and stressed conditions.